Import Everythingh Here

In [35]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq 
from langchain.agents import create_agent

LoadEnvs

In [4]:
load_dotenv()

True

LoadEv Variables

In [5]:
groq_api=os.getenv("GROQ_API_KEY")
jina_api=os.getenv("JINA_API_KEY")

loadData

In [6]:
DATA_FILE_PATH=os.path.join("data","hr_policy.txt")

DATA INGESTION

Loading

In [12]:
loader=TextLoader(DATA_FILE_PATH,encoding="utf-8")
document=loader.load()

In [13]:
print(document)

[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probatio

In [17]:
len(document)

1

data splitting

In [22]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks=text_splitter.split_documents(document)

Data Embedding

In [26]:
embedding=JinaEmbeddings(model_name='jina-embeddings-v2-base-en')

VectorStore

In [27]:
vector_store=FAISS.from_documents(chunks,embedding)

### Tools

In [37]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [ ]:
llm=ChatGroq(
    model='openai/gpt-oss-120b',
    temperature=0
)

In [41]:
hr_assistant=create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

In [42]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer